In [1]:
from pathlib import Path
import pandas as pd
from scipy.io import loadmat

In [2]:
CWRU_PATH = Path("../data/raw/CWRU")
files = sorted(CWRU_PATH.glob("*.mat"))
print("Number of files:", len(files))

Number of files: 16


In [3]:
def get_class(filename):

    if filename.startswith("NORMAL"):
        return "Healthy"

    elif filename.startswith("IR007"):
        return "Inner Race"

    elif filename.startswith("B007"):
        return "Ball"

    elif filename.startswith("OR007"):
        return "Outer Race"

    else:
        return "Unknown"

In [4]:
def get_load(filename):

    suffix = filename.split("_")[-1]
    load = int(suffix.replace(".mat", ""))

    return load

In [5]:
RPM_MAP = {
    0: 1797,
    1: 1772,
    2: 1750,
    3: 1730
}
FS = 12000

In [6]:
records = []

for file in files:

    filename = file.name

    fault_class = get_class(filename)

    load = get_load(filename)

    rpm = RPM_MAP[load]

    records.append({
        "file": filename,
        "class": fault_class,
        "load_hp": load,
        "rpm": rpm,
        "sampling_rate_hz": FS,
        "channel": "DE",
        "fault_diameter_in": 0.007
    })

In [7]:
metadata = pd.DataFrame(records)

In [8]:
metadata

,file,class,load_hp,rpm,sampling_rate_hz,channel,fault_diameter_in
0,B007_0.mat,Ball,0,1797,12000,DE,0.007
1,B007_1.mat,Ball,1,1772,12000,DE,0.007
2,B007_2.mat,Ball,2,1750,12000,DE,0.007
3,B007_3.mat,Ball,3,1730,12000,DE,0.007
4,IR007_0.mat,Inner Race,0,1797,12000,DE,0.007
5,IR007_1.mat,Inner Race,1,1772,12000,DE,0.007
6,IR007_2.mat,Inner Race,2,1750,12000,DE,0.007
7,IR007_3.mat,Inner Race,3,1730,12000,DE,0.007
8,NORMAL_0.mat,Healthy,0,1797,12000,DE,0.007
9,NORMAL_1.mat,Healthy,1,1772,12000,DE,0.007


In [9]:
print(metadata["class"].value_counts())

class
Ball          4
Inner Race    4
Healthy       4
Outer Race    4
Name: count, dtype: int64


In [10]:
print(metadata["load_hp"].value_counts().sort_index())

load_hp
0    4
1    4
2    4
3    4
Name: count, dtype: int64


In [11]:
print(metadata[metadata["class"] == "Unknown"])

Empty DataFrame
Columns: [file, class, load_hp, rpm, sampling_rate_hz, channel, fault_diameter_in]
Index: []


In [12]:
output_path = Path("../data/processed/CWRU/cwru_metadata.csv")

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

metadata.to_csv(
    output_path,
    index=False
)

In [13]:
check = pd.read_csv(
    "../data/processed/CWRU/cwru_metadata.csv"
)

print(check)

             file       class  load_hp   rpm  sampling_rate_hz channel  \
0      B007_0.mat        Ball        0  1797             12000      DE   
1      B007_1.mat        Ball        1  1772             12000      DE   
2      B007_2.mat        Ball        2  1750             12000      DE   
3      B007_3.mat        Ball        3  1730             12000      DE   
4     IR007_0.mat  Inner Race        0  1797             12000      DE   
5     IR007_1.mat  Inner Race        1  1772             12000      DE   
6     IR007_2.mat  Inner Race        2  1750             12000      DE   
7     IR007_3.mat  Inner Race        3  1730             12000      DE   
8    NORMAL_0.mat     Healthy        0  1797             12000      DE   
9    NORMAL_1.mat     Healthy        1  1772             12000      DE   
10   NORMAL_2.mat     Healthy        2  1750             12000      DE   
11   NORMAL_3.mat     Healthy        3  1730             12000      DE   
12  OR007@6_0.mat  Outer Race        0

In [14]:
print("Total recordings:", len(metadata))
print("Classes:", metadata["class"].nunique())
print("Sampling rate:", metadata["sampling_rate_hz"].unique())
print("Channels:", metadata["channel"].unique())
print("Loads:", sorted(metadata["load_hp"].unique()))

Total recordings: 16
Classes: 4
Sampling rate: [12000]
Channels: ['DE']
Loads: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
